In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df = spark.read.table(f"hcp_databricks_workspace.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
from pyspark.sql.functions import current_timestamp

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table = f"hcp_databricks_workspace.silver.{table_name}"
load_type_normalized = load_type.strip().upper()

if load_type_normalized == "FULL":
    # Full load: overwrite target table
    updated_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type_normalized == "APPEND":
    # Append load: add new rows to target table
    updated_df.write.mode("append").saveAsTable(target_table)

elif load_type_normalized == "MERGE":
    # Merge load: update existing rows and insert new rows based on merge_keys
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = 'MERGE'")

    missing_keys = [k for k in keys if k not in updated_df.columns]
    if missing_keys:
        raise ValueError(f"Merge keys not found in source data: {missing_keys}")

    # Deduplicate source on merge keys to avoid multiple-match MERGE errors
    source_df = updated_df.dropDuplicates(keys)

    if not spark.catalog.tableExists(target_table):
        # First run: create the target table
        source_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        merge_condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        # On update, keep the original insert_timestamp and refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        (
            DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute()
        )

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))